# Notebook 07 — Trading Strategy Backtest

**Goal:** test whether the model's next-period FX direction predictions would have produced a useful hypothetical trading result.

**Important limitation:** this is a transparent *illustrative backtest*, not an exact reproduction of the paper's portfolio simulation. The paper's description does not specify every operational detail clearly enough to claim exact replication. This notebook therefore states its assumptions explicitly.

**Strategy assumptions**
- Use the model's predicted next-period direction (up/down) for one FX pair at a time.
- Go long the pair when predicted direction is up; otherwise hold cash (no shorting).
- Use the prior observation's prediction to avoid using a prediction before it would have existed.
- Apply a configurable transaction cost on position changes.
- Compare against buy-and-hold and cash baselines.
- Returns are based on the pair's log returns; results are hypothetical and exclude leverage, taxes, spreads beyond the modeled cost, and market-impact effects.

Run after Notebooks 01–03. The notebook uses the existing project datasets/results when available and clearly checks required files.


## 1. Set paths and inspect available files

This notebook expects the processed pair rates and, ideally, classification predictions from Notebook 03. It can also run a small baseline strategy from the actual next-period direction if prediction files are unavailable, but that fallback is **not a valid model backtest**; it is only a wiring check. Do not report fallback results as model performance.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Results:", RESULTS_DIR)
print("Processed files:", [p.name for p in PROCESSED_DIR.glob("*")] if PROCESSED_DIR.exists() else "folder not found")
print("Result files:", [p.name for p in RESULTS_DIR.glob("*")])


## 2. Load exchange rates and classification predictions

For a valid backtest, predictions must come from the model and must be out-of-sample (validation/test), not training predictions. This cell searches for common prediction-file names. If Notebook 03 did not save per-date predictions, use the instruction printed below to add that output before proceeding.

In [ ]:
pair_rates_path = PROCESSED_DIR / "pair_rates.csv"
if pair_rates_path.exists():
    pair_rates = pd.read_csv(pair_rates_path, parse_dates=["date"], index_col="date").sort_index()
else:
    # Fallback: use raw derived dataset only if a pair_rates file exists nowhere else.
    raise FileNotFoundError(
        f"Missing {pair_rates_path}. Run Notebook 01/02, or update this cell to load your saved pair_rates dataset."
    )

pair_rates = pair_rates[["GBP_USD", "USD_EUR", "USD_JPY"]].apply(pd.to_numeric, errors="coerce").dropna()
print("Pair rates:", pair_rates.shape, pair_rates.index.min(), "to", pair_rates.index.max())

candidate_files = [
    RESULTS_DIR / "classification_test_predictions.csv",
    RESULTS_DIR / "classification_predictions.csv",
    PROCESSED_DIR / "classification_test_predictions.csv",
]
prediction_path = next((p for p in candidate_files if p.exists()), None)

if prediction_path is None:
    print(
        "\nNo per-date model prediction file found. Do NOT backtest model performance yet. "
        "Notebook 03 currently reports summary metrics; it must also save dated test predictions. "
        "The next cell explains the required format."
    )
else:
    predictions = pd.read_csv(prediction_path, parse_dates=["date"])
    print("Loaded predictions:", prediction_path)
    display(predictions.head())


## 3. Required prediction-file format

The prediction CSV should contain one row per test date and pair, with these columns:

- `date`: date of the features used to make the forecast;
- `pair`: one of `GBP_USD`, `USD_EUR`, `USD_JPY`;
- `y_true`: actual next-period direction (0 or 1), for evaluation only;
- `y_pred`: model-predicted next-period direction (0 or 1).

**Do not use `y_true` to decide the position.** It is included only for evaluation. A backtest cannot be honestly performed from aggregate accuracy alone because the sequence of predictions matters. If no dated prediction file exists, stop here and add prediction export to Notebook 03 first.


In [ ]:
if prediction_path is None:
    print("STOP: create dated test predictions from Notebook 03 before running the strategy cells.")
else:
    required = {"date", "pair", "y_true", "y_pred"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Prediction file missing columns: {missing}")
    predictions["date"] = pd.to_datetime(predictions["date"])
    predictions["pair"] = predictions["pair"].astype(str)
    predictions["y_true"] = pd.to_numeric(predictions["y_true"], errors="coerce")
    predictions["y_pred"] = pd.to_numeric(predictions["y_pred"], errors="coerce")
    predictions = predictions.dropna(subset=["date", "pair", "y_true", "y_pred"])
    predictions["y_true"] = predictions["y_true"].astype(int)
    predictions["y_pred"] = predictions["y_pred"].astype(int)
    display(predictions.groupby("pair").agg(
        rows=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        accuracy=("y_true", lambda x: np.nan)  # replaced below with explicit accuracy
    ))
    accuracy_check = predictions.assign(correct=predictions["y_true"] == predictions["y_pred"]).groupby("pair")["correct"].mean()
    display(accuracy_check.rename("prediction_accuracy").to_frame())


## 4. Backtest settings and strategy logic

A direction classifier does not predict return magnitude. We therefore use a simple long-or-cash rule:
- `y_pred = 1`: hold the currency pair for the next observation interval;
- `y_pred = 0`: hold cash;
- position is shifted by one row before multiplying by the realized return;
- transaction costs are charged when the position changes.

**Quote convention matters:** the return of `GBP_USD` is the percentage/log change in USD per GBP, while `USD_EUR` and `USD_JPY` are also interpreted as the change in the quoted pair. This simplified strategy is not a complete multi-currency conversion portfolio like the paper's described simulation.


In [ ]:
INITIAL_CAPITAL = 100.0
TRANSACTION_COST_BPS = 2.0  # illustrative cost per position change; change only with a documented assumption
COST_RATE = TRANSACTION_COST_BPS / 10_000

def backtest_one_pair(prediction_frame, pair_rates, pair, initial_capital=100.0, cost_rate=0.0002):
    p = prediction_frame[prediction_frame["pair"] == pair].copy().sort_values("date")
    if p.empty:
        return None, None

    # Prediction date is the feature date; realized return is from date t to t+1.
    rates = pair_rates[pair].sort_index()
    log_returns = np.log(rates / rates.shift(1))
    realized = log_returns.shift(-1).rename("realized_log_return")
    frame = p.set_index("date").join(realized, how="left")
    frame = frame.dropna(subset=["realized_log_return"]).copy()

    # Position chosen from the forecast on date t, held over t -> t+1.
    frame["position"] = (frame["y_pred"] == 1).astype(float)
    frame["position_change"] = frame["position"].diff().abs().fillna(frame["position"].abs())
    frame["strategy_log_return_before_cost"] = frame["position"] * frame["realized_log_return"]
    # Convert log return to simple return before applying transaction cost.
    frame["strategy_simple_return"] = np.expm1(frame["strategy_log_return_before_cost"])
    frame["strategy_simple_return_net"] = (
        (1 + frame["strategy_simple_return"]) * (1 - frame["position_change"] * cost_rate) - 1
    )
    frame["buy_hold_simple_return"] = np.expm1(frame["realized_log_return"])
    frame["strategy_equity"] = initial_capital * (1 + frame["strategy_simple_return_net"]).cumprod()
    frame["buy_hold_equity"] = initial_capital * (1 + frame["buy_hold_simple_return"]).cumprod()
    frame["cash_equity"] = initial_capital
    return frame, p

if prediction_path is not None:
    backtest_frames = {}
    summary_rows = []
    for pair in ["GBP_USD", "USD_EUR", "USD_JPY"]:
        result, _ = backtest_one_pair(
            predictions, pair_rates, pair,
            initial_capital=INITIAL_CAPITAL, cost_rate=COST_RATE
        )
        if result is None or result.empty:
            print(f"No aligned predictions for {pair}")
            continue
        backtest_frames[pair] = result
        strat_returns = result["strategy_simple_return_net"]
        bh_returns = result["buy_hold_simple_return"]
        n = len(result)
        summary_rows.append({
            "pair": pair,
            "observations": n,
            "strategy_total_return_pct": (result["strategy_equity"].iloc[-1] / INITIAL_CAPITAL - 1) * 100,
            "buy_hold_total_return_pct": (result["buy_hold_equity"].iloc[-1] / INITIAL_CAPITAL - 1) * 100,
            "cash_total_return_pct": 0.0,
            "strategy_max_drawdown_pct": ((result["strategy_equity"] / result["strategy_equity"].cummax()) - 1).min() * 100,
            "strategy_sharpe_annualized_approx": (
                strat_returns.mean() / strat_returns.std(ddof=1) * np.sqrt(252)
                if strat_returns.std(ddof=1) and n > 1 else np.nan
            ),
            "position_fraction": result["position"].mean(),
            "transaction_cost_bps": TRANSACTION_COST_BPS
        })
    backtest_summary = pd.DataFrame(summary_rows)
    display(backtest_summary)
else:
    backtest_frames = {}
    backtest_summary = pd.DataFrame()


## 5. Plot the equity curves

In [ ]:
for pair, frame in backtest_frames.items():
    ax = frame[["strategy_equity", "buy_hold_equity", "cash_equity"]].plot(
        figsize=(12, 5), title=f"{pair}: illustrative test-period equity"
    )
    ax.set_xlabel("Test date")
    ax.set_ylabel("Hypothetical capital")
    ax.grid(True, alpha=0.3)
    plt.show()


## 6. Save results and interpret carefully

The outputs are hypothetical. A positive cumulative return alone is not enough to claim a useful strategy: compare against buy-and-hold and cash, inspect drawdowns and costs, and avoid tuning the strategy on the test period.

If the notebook stopped because dated predictions were not found, that is expected: aggregate accuracy from Notebook 03 is insufficient for a valid backtest. Add dated test prediction export first; do not fabricate predictions or use actual future direction as the trading signal.


In [ ]:
if not backtest_summary.empty:
    summary_path = RESULTS_DIR / "trading_backtest_summary.csv"
    backtest_summary.to_csv(summary_path, index=False)
    print("Saved summary:", summary_path)

    for pair, frame in backtest_frames.items():
        output_path = RESULTS_DIR / f"trading_backtest_{pair}.csv"
        frame.to_csv(output_path, index=True, index_label="date")
        print("Saved:", output_path)
else:
    print("No backtest results saved. First export dated test predictions from Notebook 03.")
